## How this notebook runs

**Training runs in SageMaker training jobs** on dedicated GPU instances.

This notebook orchestrates the workflow:
- Carves stratified data splits and uploads them to S3 channels
- Runs local unit tests and throughput benchmarks
- Launches baseline and decider-v2 training jobs concurrently
- Monitors job progress, downloads checkpoints, and evaluates on the test split
- Runs JevBench and sysone-bench adapters and generates the final decision report.


## 1. Sources (embedded, self-extracting)

The training sources (`decider-train/`) are embedded in the notebook as a base64 zip and extracted to `./decider_v2_src` on first run. Extraction is validated using a content SHA256 digest.


In [ ]:
# --- 0-sources ---
SRC_ZIP_B64 = "..."

import base64, hashlib, io, os, zipfile

SRC_DIR = os.path.abspath("decider_v2_src")
STAMP = os.path.join(SRC_DIR, ".notebook_zip.sha256")
FORCE_REUSE = False

_blob = base64.b64decode(SRC_ZIP_B64)
_digest = hashlib.sha256(_blob).hexdigest()

_on_disk = None
if os.path.isfile(STAMP):
    try:
        _on_disk = open(STAMP, encoding="utf-8").read().strip()
    except OSError:
        _on_disk = None

_have_sources = os.path.isfile(os.path.join(SRC_DIR, "train.py"))
if FORCE_REUSE or (_have_sources and _on_disk == _digest):
    print(f"decider_v2_src up to date (sha256: {_digest[:12]})")
else:
    with zipfile.ZipFile(io.BytesIO(_blob)) as _z:
        _names = [n for n in _z.namelist() if not n.endswith("/")]
        for _n in _names:
            _rel = _n[len("decider_v2_src/"):] if _n.startswith("decider_v2_src/") else _n
            _dest = os.path.abspath(os.path.join(SRC_DIR, _rel))
            if not _dest.startswith(SRC_DIR + os.sep):
                raise SystemExit(f"Refusing unsafe path in embedded zip: {_n}")
            os.makedirs(os.path.dirname(_dest), exist_ok=True)
            with _z.open(_n) as _src, open(_dest, "wb") as _out:
                _out.write(_src.read())
    try:
        with open(STAMP, "w", encoding="utf-8") as f:
            f.write(_digest)
    except OSError:
        pass
    print(f"Extracted {len(_names)} files to {SRC_DIR} (sha256: {_digest[:12]})")

## 2. Configuration

Configure AWS resources, data parameters, and training budgets. Set `SMOKE = False` for the full SageMaker training run (14,688 steps / 940k presentations on 80k test set), or `SMOKE = True` for quick end-to-end verification (100 steps on 4k samples).


In [ ]:
# --- 1-config ---
import getpass, json, os, sys, time

# ------------------------------------------------------------------ User Config
BUCKET          = ""                 # S3 bucket for data & artifacts (leave empty to use SageMaker default bucket)
PREFIX          = "decider_v2"       # S3 prefix for run artifacts
REGION          = ""                 # AWS Region (leave empty to use boto3 session default)
INSTANCE_TYPE   = "ml.g5.2xlarge"    # SageMaker training instance (1x NVIDIA A10G 24GB)
ROLE_NAME       = ""                 # IAM execution role ARN (leave empty to resolve via sagemaker.get_execution_role())

FRAMEWORK_VERSION = "2.7.1"
PY_VERSION        = "py312"

SMOKE           = False              # False: full training (14,688 steps / 940k presentations); True: 100-step smoke test

BATCH_SIZE      = 32
GRAD_ACCUM      = 2
MAX_STATE_TOKENS = 1024
MAX_CHOICE_TOKENS = 64

TARGET_PRESENTATIONS = 940_000
SEEDS = [0]
SPLIT_SEED = 0

RUN_BASELINE    = True
RUN_V2          = True
RUN_ABLATIONS   = False

USE_SPOT        = True
SPOT_MAX_WAIT_MINUTES = 72 * 60
MAX_RUNTIME_HOURS = 72
MAX_RUNTIME     = MAX_RUNTIME_HOURS * 3600

ON_DEMAND_PRICE_PER_HOUR = None
SPOT_PRICE_PER_HOUR      = None

REBUILD_VENV    = False
SELFTEST        = True
ENV_CHECK       = True

# Derived variables
RECIPE_QPS      = 64
ACTUAL_QPS      = BATCH_SIZE * GRAD_ACCUM
MAX_STEPS_REAL  = TARGET_PRESENTATIONS // ACTUAL_QPS
MAX_STEPS       = 100 if SMOKE else MAX_STEPS_REAL
SMOKE_TEST_N    = 4000
SMOKE_KW        = dict(smoke_train=4000, smoke_val=400, smoke_calib=400, smoke_test=SMOKE_TEST_N) if SMOKE else {}

RUN_NAME = f"run_{time.strftime('%Y%m%d_%H%M%S')}_{getpass.getuser()}"
LOCAL = os.path.abspath("decider_v2_work")
os.makedirs(LOCAL, exist_ok=True)

PRESENTATIONS = MAX_STEPS * ACTUAL_QPS
print(f"Config: SMOKE={SMOKE} | Instance={INSTANCE_TYPE} | Steps={MAX_STEPS:,} | Target={PRESENTATIONS:,} presentations | Seeds={SEEDS}")

## 3. Environment & AWS Session Setup

Verifies environment pins, establishes the AWS boto3 session, resolves IAM execution roles, and verifies S3 bucket read/write permissions.


In [ ]:
# --- 2-install ---
import json, os, subprocess, sys
sys.path.insert(0, os.path.abspath("decider_v2_src"))
import logger as environment

REQ = os.path.join(SRC_DIR, "requirements.txt")
if not os.path.exists(REQ):
    raise SystemExit(f"{REQ} not found.")

ok, _rows = environment.check(pins=environment.parse_requirements(REQ), verbose=False)
if ok:
    print("Kernel environment satisfies all pins.")
else:
    print("Kernel environment differs from training pins; subprocess venv will isolate dependencies.")

def _try_import(name):
    try:
        return __import__(name)
    except Exception:
        return None

torch = _try_import("torch")
transformers = _try_import("transformers")
datasets = _try_import("datasets")
peft = _try_import("peft")

import boto3
from botocore.exceptions import ClientError
SESSION = boto3.Session(region_name=REGION or None)

def resolve_role():
    if ROLE_NAME:
        return ROLE_NAME
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        pass
    ident = SESSION.client("sts").get_caller_identity()
    arn = ident.get("Arn", "")
    acct = ident.get("Account", "")
    if ":assumed-role/" in arn:
        role_name = arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        return f"arn:aws:iam::{acct}:role/{role_name}"
    return arn

ROLE = resolve_role()

def resolve_bucket():
    if BUCKET:
        return BUCKET
    try:
        import sagemaker
        return sagemaker.Session(boto_session=SESSION).default_bucket()
    except Exception:
        pass
    try:
        acct = SESSION.client("sts").get_caller_identity()["Account"]
        reg = REGION or SESSION.region_name or "us-east-1"
        return f"sagemaker-{reg}-{acct}"
    except Exception:
        return "decider-v2-training"

BUCKET = resolve_bucket()

def ensure_bucket(name, region):
    s3 = SESSION.client("s3")
    region = region or SESSION.region_name or "us-east-1"
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError as e:
        status = e.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status == 403:
            raise SystemExit(f"Bucket {name!r} exists but access is denied. Check IAM permissions.")
    try:
        if region == "us-east-1":
            s3.create_bucket(Bucket=name)
        else:
            s3.create_bucket(Bucket=name, CreateBucketConfiguration={"LocationConstraint": region})
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        if code not in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            raise SystemExit(f"Could not create bucket {name!r}: {code}")
    return True

ensure_bucket(BUCKET, REGION)

def _s3_probe(name):
    s3 = SESSION.client("s3")
    key = f"{PREFIX}/_access_probe.txt"
    body = b"decider-v2 access probe"
    try:
        s3.put_object(Bucket=name, Key=key, Body=body)
        got = s3.get_object(Bucket=name, Key=key)["Body"].read()
        s3.delete_object(Bucket=name, Key=key)
        return got == body, "OK"
    except ClientError as e:
        return False, str(e)

_ok, _msg = _s3_probe(BUCKET)
if not _ok:
    raise SystemExit(f"Role {ROLE} cannot write to s3://{BUCKET}: {_msg}")

import venv_env
VENV_DIR = os.path.join(LOCAL, "venv")
LOCAL_PY = venv_env.ensure(VENV_DIR, REQ, force=REBUILD_VENV)
LOCAL_ENV = venv_env.patch_env()

print(f"Environment: torch={getattr(torch, '__version__', 'n/a')} (cuda={getattr(torch.cuda, 'is_available', lambda: False)()})")
print(f"AWS: region={REGION or SESSION.region_name} | role={ROLE} | bucket={BUCKET}")
print(f"Subprocess venv: {LOCAL_PY}")

In [ ]:
# --- 2b-state-helper ---
import json, os, pathlib

STATE_PATH = pathlib.Path("decider_v2_state.json")

def load_state():
    if STATE_PATH.exists():
        try:
            return json.loads(STATE_PATH.read_text())
        except Exception:
            return {}
    return {}

STATE = load_state()

def save_state(**kw):
    s = load_state()
    s.update(kw)
    STATE.clear()
    STATE.update(s)
    s.setdefault("run_name", RUN_NAME if "RUN_NAME" in dir() else s.get("run_name"))
    STATE_PATH.write_text(json.dumps(s, indent=2, default=str))
    return STATE

## 4. Dataset Preparation & Parity Checks

Loads `avbiswas/bev-decision('all')`, carves train/val/calib/test splits, checks state-isolation parity, and uploads the parquet shards to S3 input channels.


In [ ]:
# --- 3-load ---
import collections, json, os, sys
CACHE = os.path.join(LOCAL, "hf_cache")
os.makedirs(CACHE, exist_ok=True)

try:
    from datasets import load_dataset
    print("Loading avbiswas/bev-decision config='all'...")
    ds = load_dataset("avbiswas/bev-decision", "all", cache_dir=CACHE)
    print(f"Dataset loaded: train={len(ds['train']):,} | test={len(ds['test']):,}")
except Exception as e:
    print(f"In-kernel dataset load skipped ({type(e).__name__}: {e}); subprocess prepare will handle it.")
    ds = None

In [ ]:
# --- 3-stats ---
import collections, json
if ds is not None:
    type_counts = collections.Counter()
    for row in ds["train"]:
        for q in json.loads(row["questions_json"]).values():
            type_counts[q["type"]] += 1
    total_q = sum(type_counts.values())
    print("Question types (train): " + ", ".join(f"{k}: {v:,} ({v/total_q*100:.1f}%)" for k, v in sorted(type_counts.items())))

In [ ]:
# --- 3-prepare ---
import subprocess
print("Carving stratified splits into Parquet shards...")
subprocess.check_call([LOCAL_PY, os.path.join(SRC_DIR, "dataset.py"), "prepare",
                       "--out", os.path.join(LOCAL, "data"), "--seed", str(SPLIT_SEED),
                       "--calib_target", "20000", "--val_frac", "0.03",
                       "--cache_dir", CACHE, "--force"]
                      + [f"--{k}={v}" for k, v in SMOKE_KW.items() if v], env=LOCAL_ENV)
m = json.load(open(os.path.join(LOCAL, "data", "manifest.json")))
print(f"Prepared splits: {m['sizes']}")

In [ ]:
# --- 3-parity ---
m = json.load(open(os.path.join(LOCAL, "data", "manifest.json")))
assert m["state_overlap_val_to_test"] == 0, "val overlaps test!"
assert m["state_overlap_calib_to_test"] == 0, "calib overlaps test!"
n_test = m["sizes"]["test"]
if not SMOKE:
    assert n_test == 80019, f"Real run requires all 80,019 test questions, got {n_test}"

TEST_N = n_test
save_state(manifest=m, data_dir=os.path.join(LOCAL, "data"), test_n=TEST_N, split_seed=SPLIT_SEED)
print(f"Split parity verified: overlap=0 | test_n={TEST_N:,} ({'SMOKE shard' if SMOKE else 'FULL published test split'})")

In [ ]:
# --- 3-upload ---
s3 = SESSION.client("s3")
DATA_PREFIX = f"{PREFIX}/{RUN_NAME}/data"
uploaded = []
for split in ("train", "val", "calib", "test"):
    key = f"{DATA_PREFIX}/{split}/questions.parquet"
    src = os.path.join(LOCAL, "data", split, "questions.parquet")
    s3.upload_file(src, BUCKET, key)
    uploaded.append(key)

save_state(data_prefix=DATA_PREFIX, channels={s: f"{DATA_PREFIX}/{s}" for s in ("train", "val", "calib", "test")})
print(f"Uploaded {len(uploaded)} input channels to s3://{BUCKET}/{DATA_PREFIX}/")

## 5. Source Packaging & Unit Testing

Archives training sources for the SageMaker estimator and executes `tests/test_core.py` locally.


In [ ]:
# --- 4-source ---
import shutil, tarfile
src_tar = os.path.join(LOCAL, "sourcedir.tar.gz")
with tarfile.open(src_tar, "w:gz") as tar:
    tar.add(SRC_DIR, arcname=".")
print(f"Packaged training code: {src_tar}")

In [ ]:
# --- 5-tests ---
import subprocess
print("Running decider-train tests/test_core.py...")
# The device choice must come from the interpreter that RUNS the tests (the pinned venv), not
# from this kernel process: a GPU torch in the kernel + a CPU torch in the venv answered this
# 'cuda' and made the battery assert into the no-CUDA build.
try:
    cuda_ok = subprocess.check_output([LOCAL_PY, "-c", "import torch; print(torch.cuda.is_available())"],
                                       env=LOCAL_ENV, text=True).strip() == "True"
except Exception:
    cuda_ok = False
subprocess.check_call([LOCAL_PY, os.path.join(SRC_DIR, "tests", "test_core.py"),
                       "--device", "cuda" if cuda_ok else "cpu",
                       "--skip_overfit"], cwd=SRC_DIR, env=LOCAL_ENV)
print("All preflight unit tests PASSED.")

## 6. Throughput Benchmark & Budget Estimation

Measures training throughput on the local card (if GPU available), calculates projected training costs, and verifies `max_runtime` headroom.


In [ ]:
# --- 6-smoke ---
import json, os, subprocess, torch

if not torch.cuda.is_available():
    print("Throughput bench: skipped (no GPU on notebook instance)")
    BENCH = None
    save_state(bench=None, gpu_available=False)
else:
    BENCH_PATH = os.path.join(LOCAL, "bench.json")
    subprocess.check_call([LOCAL_PY, os.path.join(SRC_DIR, "scripts", "bench.py"),
                           "--train_channel", os.path.join(LOCAL, "data", "train"),
                           "--num_layers", "20", "--head", "v2", "--lora_r", "16",
                           "--lora_include_mlp", "1", "--batch_size", str(min(BATCH_SIZE, 8)),
                           "--micro_steps", "6", "--max_state_tokens", str(MAX_STATE_TOKENS),
                           "--gradient_checkpointing", "1", "--num_workers", "2",
                           "--out", BENCH_PATH], env=LOCAL_ENV)
    BENCH = json.load(open(BENCH_PATH))
    save_state(bench=BENCH)
    print(f"Throughput: {BENCH['gpu_only_qps']:.1f} q/s (peak VRAM: {BENCH['peak_vram_gib']:.2f} GiB)")

In [ ]:
# --- 6-estimate ---
BENCH = STATE.get("bench")
N_ARMS = len(SEEDS) * (int(RUN_BASELINE) + int(RUN_V2))
QUESTIONS = PRESENTATIONS

if BENCH:
    v2_qps = BENCH["gpu_only_qps"]
    base_qps = v2_qps * 1.32
    print(f"Estimated runtime (budget: {QUESTIONS:,} presentations):")
    print(f"  baseline: ~{QUESTIONS / base_qps / 3600:.1f}h per job ({base_qps:.1f} q/s)")
    print(f"  v2:       ~{QUESTIONS / v2_qps / 3600:.1f}h per job ({v2_qps:.1f} q/s)")
else:
    print(f"Budget: {QUESTIONS:,} presentations across {N_ARMS} training job(s).")

In [ ]:
# --- 6b-runtime-check ---
BENCH = STATE.get("bench")
_hours = MAX_RUNTIME / 3600
if BENCH:
    qps = BENCH["gpu_only_qps"]
    h = QUESTIONS / qps / 3600
    if _hours - h <= 0 and not SMOKE:
        raise SystemExit(f"Job runtime {h:.1f}h exceeds max_runtime {MAX_RUNTIME_HOURS}h.")
    print(f"max_runtime check: PASS ({MAX_RUNTIME_HOURS}h limit > {h:.1f}h estimated job runtime)")
else:
    print(f"max_runtime configured to {MAX_RUNTIME_HOURS}h.")

## 7. Published Reference Evaluation

Evaluates the published `avbiswas/bev-decider-0.4B` checkpoint on our test split to establish the benchmark comparison anchor.


In [ ]:
# --- 7-reference ---
import os, subprocess, torch

eval_ref_out = os.path.join(LOCAL, "eval", "metrics_reference.json")
if os.path.exists(eval_ref_out):
    print(f"Reference model already evaluated ({eval_ref_out}) -- skipping.")
else:
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    is_cpu = not torch.cuda.is_available()
    print(f"Evaluating published reference model (avbiswas/bev-decider-0.4B) on {'CPU' if is_cpu else 'GPU'}...")
    if is_cpu and not SMOKE:
        print("Note: Evaluating 80,019 questions on CPU takes significant time. Training jobs in Cells 8 & 9")
        print("can be launched without waiting for this cell to finish.")

    subprocess.check_call([LOCAL_PY, "-u", os.path.join(SRC_DIR, "evaluate.py"),
                           "--reference", "avbiswas/bev-decider-0.4B",
                           "--test_channel", os.path.join(LOCAL, "data", "test"),
                           "--out", os.path.join(LOCAL, "eval"),
                           "--batch_size", "4" if is_cpu else "8",
                           "--invariance", "0",
                           "--predictions_out", os.path.join(LOCAL, "eval", "predictions_reference.csv"),
                           "--tag", "reference",
                           "--env_check", "0"], cwd=SRC_DIR, env=env)

In [ ]:
# --- 7b-reference-default ---
import subprocess
try:
    from dataset import explode_questions
    from datasets import load_dataset
    out_dir = os.path.join(LOCAL, "data", "test_default")
    os.makedirs(out_dir, exist_ok=True)
    if not os.path.exists(os.path.join(out_dir, "questions.parquet")):
        d = load_dataset("avbiswas/bev-decision", "default", cache_dir=CACHE)
        t = d["test"].add_column("row_index", list(range(len(d["test"]))))
        t = t.map(explode_questions, batched=True, batch_size=1000, remove_columns=d["test"].column_names)
        import random
        ids = sorted(random.Random(SPLIT_SEED).sample(range(len(t)), min(5000, len(t))))
        t.select(ids).to_parquet(os.path.join(out_dir, "questions.parquet"))
    subprocess.check_call([LOCAL_PY, os.path.join(SRC_DIR, "evaluate.py"),
                           "--reference", "avbiswas/bev-decider-0.4B",
                           "--test_channel", out_dir, "--out", os.path.join(LOCAL, "eval"),
                           "--batch_size", "8", "--invariance", "0", "--tag", "reference_default",
                           "--env_check", "0"], cwd=SRC_DIR, env=LOCAL_ENV)
except Exception as e:
    print(f"Reference default-config reproduction skipped: {e}")

## 8. Launch Baseline Replica

Constructs the SageMaker estimator and launches the baseline training jobs.

**AWS Quota Check:** Ensure your region has quota for `ml.g5.2xlarge` (`aws service-quotas list-service-quotas --service-code sagemaker`).


In [ ]:
# --- 8-launch-baseline ---
from sagemaker.pytorch import PyTorch

METRIC_DEFINITIONS = [
    {"Name": "train:loss", "Regex": r"step=\d+\s+loss=([0-9\.]+)"},
    {"Name": "train:grad_norm", "Regex": r"grad_norm=([0-9\.]+)"},
    {"Name": "train:acc", "Regex": r"acc=([0-9\.]+)"},
    {"Name": "train:sps", "Regex": r"sps=([0-9\.]+)"},
    {"Name": "val:loss", "Regex": r"val_loss=([0-9\.]+)"},
    {"Name": "val:acc", "Regex": r"val_acc=([0-9\.]+)"},
]

def hyper(seed, **over):
    h = dict(
        max_state_tokens=MAX_STATE_TOKENS, max_choice_tokens=MAX_CHOICE_TOKENS,
        num_layers=20, mode="bev", pool="bev",
        head="v1", head_new_dim=512, head_layers=2,
        lora_r=8, lora_alpha=16, lora_last_k_layers=12, lora_include_mlp=0,
        label_smoothing=0.0, emd_lambda=0.0, type_weights="",
        lr_lora=1e-4, lr_head=1e-3, warmup_ratio=0.05, weight_decay=0.01, grad_clip=1.0,
        shuffle_options=0, gradient_checkpointing=1,
        batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM, max_steps=MAX_STEPS, epochs=2,
        seed=seed, num_workers=2 if SMOKE else 8,
        log_every=max(5, MAX_STEPS // 60), eval_every=max(20, MAX_STEPS // 4),
        save_every=max(20, MAX_STEPS // 4), max_train_minutes=-1,
        selftest=int(bool(SELFTEST)), env_check=int(bool(ENV_CHECK)),
    )
    h.update({k: str(v) for k, v in over.items()})
    return [f"--{k}={v}" for k, v in h.items()]

def launch(name, args, channels, wait=False):
    key = f"jobs/{name}"
    if STATE.get(key) and STATE[key].get("status") in ("Completed", "Failed", "Stopped"):
        print(f"Job {name} already {STATE[key]['status']} -- skipping.")
        return STATE[key]["name"]

    ckpt_s3 = f"s3://{BUCKET}/{PREFIX}/{RUN_NAME}/{name}/checkpoints"
    max_wait = SPOT_MAX_WAIT_MINUTES * 60 if USE_SPOT else None

    # Pass individual hyperparameters for SageMaker UI tracking + sagemaker_args for CLI
    if isinstance(args, dict):
        hp = {k: str(v) for k, v in args.items()}
        hp["sagemaker_args"] = " ".join(f"--{k}={v}" for k, v in args.items())
    else:
        hp = {"sagemaker_args": " ".join(args)}
        for a in args:
            if a.startswith("--") and "=" in a:
                k, v = a[2:].split("=", 1)
                hp[k] = v

    est = PyTorch(
        entry_point="train.py", source_dir=SRC_DIR, role=ROLE,
        framework_version=FRAMEWORK_VERSION, py_version=PY_VERSION,
        instance_type=INSTANCE_TYPE, instance_count=1,
        output_path=f"s3://{BUCKET}/{PREFIX}/{RUN_NAME}/{name}/output",
        checkpoint_s3_uri=ckpt_s3,
        hyperparameters=hp,
        metric_definitions=METRIC_DEFINITIONS,
        input_channels=channels, volume_size=200, keep_alive_period_in_seconds=600,
        use_spot=USE_SPOT, max_runtime=MAX_RUNTIME,
    )
    if USE_SPOT:
        est.configure(WaitTimeConfig={"MaxWaitTime": max_wait})
    est.fit(wait=wait, logs="None")
    print(f"Launched {name} -> SageMaker job {est._latest_job_name}")
    save_state(**{key: {"name": est._latest_job_name, "status": "InProgress",
                        "output": f"{PREFIX}/{RUN_NAME}/{name}/output",
                        "checkpoints": f"{PREFIX}/{RUN_NAME}/{name}/checkpoints"}})
    return est._latest_job_name

CHANNELS = {k: f"s3://{BUCKET}/{v}" for k, v in STATE["channels"].items()}
JOBS = {}
for seed in SEEDS:
    if RUN_BASELINE:
        launch(f"baseline_s{seed}", hyper(seed), CHANNELS)
        JOBS.setdefault("baseline", []).append(seed)
print(f"Baseline jobs: {JOBS.get('baseline', [])}")

## 9. Launch decider-v2

Launches the decider-v2 jobs with Tier-1 architectural additions: Perceiver context summarizer, level-traveling rank embedding, and LoRA on all 20 layers.


In [ ]:
# --- 9-launch-v2 ---
V2_OVER = dict(
    head="v2", head_new_dim=512, head_layers=2, head_heads=8, n_ctx=8,
    use_ctx=1, use_rank=1, use_null=0, use_derived=0,
    lora_r=16, lora_alpha=32, lora_last_k_layers=20, lora_include_mlp=1,
    label_smoothing=0.05, emd_lambda=0.5, type_weights="",
    lr_lora=2e-4, lr_head=1e-3, warmup_ratio=0.03, shuffle_options=1,
)
for seed in SEEDS:
    if RUN_V2:
        launch(f"v2_s{seed}", hyper(seed, **V2_OVER), CHANNELS)
        JOBS.setdefault("v2", []).append(seed)
print(f"decider-v2 jobs: {JOBS.get('v2', [])}")
save_state(jobs_by_arm=JOBS)

## 10. Job Monitoring & Training Curves

Polls running SageMaker training jobs, downloads completed checkpoints, and plots training loss / validation accuracy.


In [ ]:
# --- 10-monitor ---
import glob, json, os, tarfile, time
from sagemaker import Session as SMSession
sm = SMSession(boto_session=SESSION)

def job_status(name):
    try:
        d = sm.describe_training_job(TrainingJobName=STATE[f"jobs/{name}"]["name"])
        return d["TrainingJobStatus"]
    except Exception as e:
        return f"unknown ({type(e).__name__})"

def download_outputs(name):
    j = STATE.get(f"jobs/{name}")
    if not j:
        return []
    bucket = boto3.resource("s3", region_name=REGION or None).Bucket(BUCKET)
    local = os.path.join(LOCAL, "downloads", name)
    files = []
    for prefix in (j["output"], j.get("checkpoints", "")):
        if not prefix:
            continue
        for obj in bucket.objects.filter(Prefix=prefix):
            rel = os.path.relpath(obj.key, f"{PREFIX}/{RUN_NAME}/{name}").replace("\\", "/")
            target = os.path.join(local, rel)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            bucket.download_file(obj.key, target)
            files.append(target)
            if target.endswith(".tar.gz"):
                try:
                    with tarfile.open(target, "r:gz") as tar:
                        tar.extractall(path=os.path.dirname(target))
                except Exception as e:
                    print(f"Failed extracting {target}: {e}")
    return files

WAIT = True           # Wait for training jobs to complete before proceeding to evaluation
POLL_INTERVAL = 30    # seconds

todo = sorted(k[len("jobs/"):] for k in STATE if k.startswith("jobs/"))
print(f"Monitoring {len(todo)} job(s): {todo}")

while True:
    statuses = {n: job_status(n) for n in todo}
    active = [n for n, s in statuses.items() if s not in ("Completed", "Failed", "Stopped")]
    status_str = " | ".join(f"{n}: {statuses[n]}" for n in todo)
    print(f"[{time.strftime('%H:%M:%S')}] {status_str}")

    for n, st in statuses.items():
        if st in ("Completed", "Stopped") and not STATE.get(f"downloaded/{n}"):
            print(f"Downloading outputs for completed job {n}...")
            download_outputs(n)
            save_state(**{f"downloaded/{n}": True})
        save_state(**{f"jobs/{n}": {**STATE[f"jobs/{n}"], "status": st}})

    if not active or not WAIT:
        break
    time.sleep(POLL_INTERVAL)

In [ ]:
# --- 10-plot ---
import glob, json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = {"baseline": "tab:blue", "v2": "tab:orange"}

plotted = False
for name in sorted(k[len("jobs/"):] for k in STATE if k.startswith("jobs/")):
    arm = name.split("_")[0]
    colour = palette.get(arm, "tab:green")
    matches = glob.glob(os.path.join(LOCAL, "downloads", name, "**", "metrics.jsonl"), recursive=True)
    if not matches:
        continue
    recs = [json.loads(line) for line in open(matches[0])]
    tr = [r for r in recs if "loss" in r]
    va = [r for r in recs if "val_acc" in r]
    if tr:
        axes[0].plot([r["step"] for r in tr], [r["loss"] for r in tr], color=colour, label=name)
        plotted = True
    if va:
        axes[1].plot([r["step"] for r in va], [r["val_acc"] for r in va], "o-", color=colour, label=name, ms=3)
        plotted = True

if plotted:
    axes[0].set(xlabel="step", ylabel="train loss", title="Training Loss")
    axes[1].set(xlabel="step", ylabel="val acc", title="Validation Accuracy")
    for a in axes:
        a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(LOCAL, "training_curves.png"), dpi=120)
    print("Saved training curves to decider_v2_work/training_curves.png")
else:
    print("No training curves to plot yet (jobs still running).")

## 11. Calibration & Test Split Evaluation

Calibrates per-type temperatures on `calib` split, then evaluates accuracy, ECE, NLL, and option-order invariance on the held-out `test` split.


In [ ]:
# --- 11-calibrate-evaluate ---
import glob, os, subprocess, torch

evaluated = []
for name in sorted(k[len("jobs/"):] for k in STATE if k.startswith("jobs/")):
    if STATE[f"jobs/{name}"].get("status") != "Completed":
        continue
    local = os.path.join(LOCAL, "downloads", name)
    ckpt = None
    for root, _dirs, files in os.walk(local):
        if "model.safetensors" in files:
            if os.path.basename(root) == "best" or ckpt is None:
                ckpt = root
    if not ckpt:
        continue

    out = os.path.join(LOCAL, "eval", name)
    os.makedirs(out, exist_ok=True)
    print(f"Evaluating {name} from {ckpt}...")
    subprocess.check_call([LOCAL_PY, os.path.join(SRC_DIR, "evaluate.py"),
                           "--checkpoint", ckpt,
                           "--test_channel", os.path.join(LOCAL, "data", "test"),
                           "--calib_channel", os.path.join(LOCAL, "data", "calib"),
                           "--out", out,
                           "--batch_size", "4" if not torch.cuda.is_available() else "8",
                           "--invariance", "1", "--invariance_examples", "32",
                           "--predictions_out", os.path.join(out, f"predictions_{name}.csv"),
                           "--tag", name,
                           "--env_check", "0"], cwd=SRC_DIR, env=LOCAL_ENV)
    evaluated.append(name)

save_state(evaluated=evaluated)
print(f"Completed evaluation for: {evaluated}")

## 12. Standard Benchmarks (JevBench & sysone-bench)

Runs standard downstream benchmarks if cloned checkouts are pointed to by `JEVBENCH_ROOT` and `SYSONE_BENCH_ROOT`.


In [ ]:
# --- 12-benchmarks ---
import json, os, subprocess

JEVBENCH_ROOT = os.environ.get("JEVBENCH_ROOT", "")
SYSONE_ROOT   = os.environ.get("SYSONE_BENCH_ROOT", "")
BENCH_DIR     = os.path.join(SRC_DIR, "benchmarks")

missing = [n for n, p in (("JEVBENCH_ROOT", JEVBENCH_ROOT), ("SYSONE_BENCH_ROOT", SYSONE_ROOT))
           if not p or not os.path.isdir(p)]
if missing:
    print(f"Standard benchmarks skipped (optional repos unset: {missing})")
else:
    for target in ["reference"] + STATE.get("evaluated", []):
        for script, rootpath in (("run_jevbench.py", JEVBENCH_ROOT), ("run_sysone_bench.py", SYSONE_ROOT)):
            out_json = os.path.join(LOCAL, "eval", "bench", f"{target}_{script[:-3]}.json")
            os.makedirs(os.path.dirname(out_json), exist_ok=True)
            subprocess.call([LOCAL_PY, os.path.join(BENCH_DIR, script),
                             "--batch_size", "8", "--out", out_json,
                             "--jevbench_root" if "jev" in script else "--sysone_bench_root", rootpath],
                            env=LOCAL_ENV)
    print("Benchmark evaluations complete.")

## 13. Results & Decision Analysis

Aggregates test split metrics, performs paired bootstrap hypothesis testing, evaluates the four separation criteria, and writes `RESULTS.md`.


In [ ]:
# --- 13-results ---
import json, os, sys

def load(tag):
    for path in (os.path.join(LOCAL, "eval", tag, f"metrics_{tag}.json"),
                 os.path.join(LOCAL, "eval", f"metrics_{tag}.json")):
        if os.path.exists(path):
            return json.load(open(path))
    return None

def overall(m):
    c, by_type = m["count_by_type"], m["accuracy_by_type"]
    return sum(by_type[k] * c[k] for k in c) / sum(c.values())

evaluated = STATE.get("evaluated", [])
w = "{:<40s} {:>8s} {:>8s} {:>8s} {:>8s} {:>8s}"
print(w.format("Model", "All", "Choice", "Noul", "Score", "ECE"))
print("-" * 84)

rows = []
for tag in ["reference"] + evaluated:
    mm = load(tag)
    if not mm:
        continue
    acc = overall(mm) * 100
    t = mm["accuracy_by_type"]
    print(w.format(tag, f"{acc:.2f}%", f"{t['choice']*100:.2f}%", f"{t['noul']*100:.2f}%", f"{t['score']*100:.2f}%", f"{mm['ece']:.4f}"))
    rows.append((tag, acc, t, mm["ece"]))

# Statistical decision rule via decider-train/decide.py
sys.path.insert(0, SRC_DIR)
import decide
res = decide.analyse(os.path.join(LOCAL, "eval"), "baseline", "v2", seeds=[0])
if res:
    print("\n" + decide.format_report(res))
    save_state(decision=res)

# Write RESULTS.md
res_lines = ["# decider-v2 Experiment Results", "", f"Run: {RUN_NAME}", ""]
res_lines.append("| Model | Accuracy | Choice | Noul | Score | ECE |")
res_lines.append("|---|---|---|---|---|---|")
for tag, acc, t, ece in rows:
    res_lines.append(f"| {tag} | {acc:.2f}% | {t['choice']*100:.2f}% | {t['noul']*100:.2f}% | {t['score']*100:.2f}% | {ece:.4f} |")

if res:
    res_lines.extend(["", "## Statistical Verdict", f"**Verdict: {res['verdict']}** (Delta: {res['delta_pp']:+.2f} pp, 95% CI: [{res['ci_low_pp']:+.2f}, {res['ci_high_pp']:+.2f}])"])

with open("RESULTS.md", "w", encoding="utf-8") as f:
    f.write("\n".join(res_lines) + "\n")
print("Saved summary report to RESULTS.md")

In [ ]:
# --- 13-charts ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tags = ["reference"] + [t for t in STATE.get("evaluated", []) if load(t)]
scores = [overall(load(t)) * 100 for t in tags]
if scores:
    fig, ax = plt.subplots(figsize=(max(5, len(tags) * 1.5), 4))
    bars = ax.bar(tags, scores, color=["#7f7f7f", "#1f77b4", "#ff7f0e"][:len(tags)])
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Test Accuracy Comparison")
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig("RESULTS_by_type.png", dpi=120)
    print("Saved accuracy chart to RESULTS_by_type.png")

## 14. Optional Ablations

Runs optional architectural ablations (narrow LoRA, 24 layers, mean pooling, derived features) with seed 0.


In [ ]:
# --- 14-ablations ---
ABLATIONS = {
    "v2_lora_bev": dict(lora_include_mlp=0, lora_last_k_layers=12, lora_r=8, lora_alpha=16),
    "v2_layers24": dict(num_layers=24, lora_last_k_layers=24),
}
if RUN_ABLATIONS:
    for ab, over in ABLATIONS.items():
        print(f"Launching ablation {ab}...")
        launch(f"{ab}_s0", hyper(0, **{**V2_OVER, **over}), CHANNELS)
else:
    print("Ablations skipped (RUN_ABLATIONS=False)")

## 15. Cleanup & S3 Inventory

Inventories artifacts generated in S3 and provides optional job cleanup.


In [ ]:
# --- 15-cleanup ---
try:
    s3 = SESSION.client("s3")
    objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/{RUN_NAME}/").get("Contents", [])
    total_bytes = sum(o["Size"] for o in objs)
    print(f"S3 Artifacts: {len(objs)} objects, {total_bytes / 2**20:.1f} MiB under s3://{BUCKET}/{PREFIX}/{RUN_NAME}/")
except Exception as e:
    print(f"S3 inventory: {e}")
print("Workflow complete.")